# AI Trading Model Training - Advanced Transformer-LSTM

This notebook provides GPU access for training the **Advanced Transformer-LSTM** trading model.

## Model Configurations Available:
- **`advanced`**: Transformer-LSTM hybrid (~15M params) - Highest accuracy
- **`lightweight`**: Transformer-only (~3M params) - Balanced speed/accuracy  
- **`default`**: Simple LSTM (~245K params) - Fast, baseline

**Recommended**: Use `advanced` configuration for maximum trading accuracy.

**IMPORTANT**: This notebook contains MINIMAL code. All model logic is in `../src/` modules.

## GitHub Repository Setup

Configure your GitHub credentials to clone and access your repository.

In [ ]:
# GitHub Configuration
GITHUB_USERNAME = "tr4m0ryp"  # Replace with your GitHub username
GITHUB_TOKEN = ""  # Replace with your GitHub Personal Access Token
GITHUB_REPO_URL = "https://github.com/tr4m0ryp/GMGN_TradingBot.git"  # Replace with your repo URL

print("✓ GitHub credentials configured")
print(f"Username: {GITHUB_USERNAME}")
print(f"Repository: {GITHUB_REPO_URL}")

In [ ]:
import os
import subprocess

# Set working directory
os.chdir('/content')

# Extract repo name from URL
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
repo_path = f'/content/{repo_name}'

# Check if repo already exists
if os.path.exists(repo_path):
    print(f"⚠️  Repository '{repo_name}' already exists at {repo_path}")
    print("Skipping clone. Use the 'Git Pull' cell below to update.")
else:
    # Clone the repository with authentication
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')
    
    try:
        result = subprocess.run(
            ['git', 'clone', clone_url],
            capture_output=True,
            text=True,
            check=True
        )
        print(f"✓ Successfully cloned repository to {repo_path}")
        print(f"\nClone output:\n{result.stdout}")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error cloning repository:\n{e.stderr}")
        raise

# Add repo to Python path
import sys
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"\n✓ Added {repo_path} to Python path")

# Navigate into the repo
os.chdir(repo_path)
print(f"\n✓ Current directory: {os.getcwd()}")
print(f"\nRepository contents:")
for item in os.listdir('.'):
    print(f"  - {item}")

## Git Pull (Update Repository)

Run this cell to pull the latest changes from the repository.

In [ ]:
import os
import subprocess

# Ensure we're in the repo directory
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
repo_path = f'/content/{repo_name}'

if not os.path.exists(repo_path):
    print("✗ Repository not found. Please run the clone cell above first.")
else:
    os.chdir(repo_path)
    
    try:
        # Fetch latest changes
        result = subprocess.run(
            ['git', 'pull'],
            capture_output=True,
            text=True,
            check=True
        )
        print("✓ Successfully pulled latest changes")
        print(f"\nGit output:\n{result.stdout}")
        
        # Show current branch and commit
        branch_result = subprocess.run(
            ['git', 'branch', '--show-current'],
            capture_output=True,
            text=True
        )
        
        commit_result = subprocess.run(
            ['git', 'log', '-1', '--oneline'],
            capture_output=True,
            text=True
        )
        
        print(f"\nCurrent branch: {branch_result.stdout.strip()}")
        print(f"Latest commit: {commit_result.stdout.strip()}")
        
    except subprocess.CalledProcessError as e:
        print(f"✗ Error pulling changes:\n{e.stderr}")

In [ ]:
# Force reload all modules after git pull
# This is necessary because Python caches imported modules

import sys

# Remove all previously imported modules from our project
modules_to_remove = [key for key in sys.modules.keys() if 'config' in key or 'models' in key or 'training' in key or 'evaluation' in key or 'data' in key or 'utils' in key]

for module in modules_to_remove:
    del sys.modules[module]

print(f"Removed {len(modules_to_remove)} cached modules")
print("Modules cleared. Run the import cell again to reload fresh code.")

## GPU Statistics Monitor

Monitor GPU usage in real-time during training.

In [ ]:
import torch
import subprocess
import time
from IPython.display import clear_output

def get_gpu_stats():
    """Get detailed GPU statistics."""
    if not torch.cuda.is_available():
        return "No GPU available"
    
    stats = []
    stats.append("=" * 70)
    stats.append("GPU STATISTICS")
    stats.append("=" * 70)
    
    # GPU Info
    stats.append(f"\nGPU Name: {torch.cuda.get_device_name(0)}")
    stats.append(f"CUDA Version: {torch.version.cuda}")
    stats.append(f"PyTorch Version: {torch.__version__}")
    
    # Memory stats
    stats.append("\n" + "-" * 70)
    stats.append("MEMORY USAGE")
    stats.append("-" * 70)
    
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    free = total_memory - reserved
    
    stats.append(f"Total Memory:     {total_memory:.2f} GB")
    stats.append(f"Allocated:        {allocated:.2f} GB ({allocated/total_memory*100:.1f}%)")
    stats.append(f"Reserved:         {reserved:.2f} GB ({reserved/total_memory*100:.1f}%)")
    stats.append(f"Free:             {free:.2f} GB ({free/total_memory*100:.1f}%)")
    
    # Peak memory usage
    peak_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
    peak_reserved = torch.cuda.max_memory_reserved(0) / 1024**3
    
    stats.append("\n" + "-" * 70)
    stats.append("PEAK USAGE")
    stats.append("-" * 70)
    stats.append(f"Peak allocated:   {peak_allocated:.2f} GB")
    stats.append(f"Peak reserved:    {peak_reserved:.2f} GB")
    
    stats.append("\n" + "=" * 70)
    
    return "\n".join(stats)

def monitor_gpu(duration=10, interval=2):
    """Monitor GPU stats for a specified duration."""
    end_time = time.time() + duration
    
    while time.time() < end_time:
        clear_output(wait=True)
        print(get_gpu_stats())
        print(f"\nMonitoring... (refreshing every {interval}s)")
        time.sleep(interval)

# Display current GPU stats
print(get_gpu_stats())

# Uncomment to monitor for 60 seconds:
# monitor_gpu(duration=60, interval=2)

In [ ]:
import os

# Verify data files exist in the cloned repository
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
data_path = f'/content/{repo_name}/ai_model/data/processed'

print(f"Checking for data files in: {data_path}")

if os.path.exists(data_path):
    files_found = os.listdir(data_path)
    print(f"✓ Data directory found")
    print(f"  Files: {files_found}")
    
    # Verify required files
    required_files = ['train_samples.pkl', 'val_samples.pkl', 'test_samples.pkl', 'metadata.pkl']
    missing_files = [f for f in required_files if f not in files_found]
    
    if missing_files:
        print(f"\n⚠️  Missing required files: {missing_files}")
    else:
        print(f"\n✓ All required data files present")
else:
    print(f"✗ Data directory not found: {data_path}")
    print("  Make sure the repository contains the ai_model/data/processed/ folder with data files")

In [ ]:
import os
import sys
import torch
from torch.utils.data import DataLoader

# Ensure we're using the correct path to the ai_model/src directory
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
src_path = f'/content/{repo_name}/ai_model/src'

print(f"Adding to Python path: {src_path}")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Change to ai_model directory for relative imports
ai_model_path = f'/content/{repo_name}/ai_model'
os.chdir(ai_model_path)
print(f"Changed directory to: {os.getcwd()}")

try:
    from config import get_config, get_advanced_config, get_lightweight_config
    from utils import set_seed, get_device, count_parameters
    from data import load_preprocessed_datasets, collate_variable_length
    from models import VariableLengthLSTMTrader
    from models.transformer_lstm import AdvancedTransformerLSTMTrader, LightweightTransformerTrader
    from training import train_model, create_model
    from evaluation import evaluate_model, comprehensive_backtest
    
    print("\n All modules imported successfully!")
    print("\nAvailable model configurations:")
    print("  - get_config('default')    : Simple LSTM (~245K params)")
    print("  - get_config('advanced')   : Transformer-LSTM (~15M params) [RECOMMENDED]")
    print("  - get_config('lightweight'): Transformer-only (~3M params)")
    print("\nYou can now proceed with the training cells below.")
    
except ModuleNotFoundError as e:
    print(f"Import failed: {e}")
    print(f"\nPlease verify that {src_path} contains all required modules.")
    print("Expected modules: config.py, utils.py, data.py, models.py, training.py, evaluation.py")

In [ ]:
# ============================================================
# MODEL CONFIGURATION SELECTION
# ============================================================
# Choose your model configuration:
#   - 'advanced'    : Transformer-LSTM hybrid (~15M params) - HIGHEST ACCURACY
#   - 'lightweight' : Transformer-only (~3M params) - Balanced
#   - 'default'     : Simple LSTM (~245K params) - Fast baseline
# ============================================================

MODEL_CONFIG = 'advanced'  # Change this to select model type

# ============================================================

# Get configuration
config = get_config(MODEL_CONFIG)

# Set random seed for reproducibility
set_seed(42)

# Get device
device = get_device()
print(f"Using device: {device}")
print(f"Model configuration: {MODEL_CONFIG}")
print(f"Model type: {config['model']['type']}")

if device == 'cuda':
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    
    # Display initial GPU memory
    total_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Total GPU Memory: {total_memory:.2f} GB")
    
print(f"\nTraining configuration:")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Epochs: {config['training']['epochs']}")
print(f"  Use cosine schedule: {config['training'].get('use_cosine_schedule', False)}")

## 1. Load Data

In [ ]:
# Load preprocessed datasets
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
data_dir = f'/content/{repo_name}/ai_model/data/processed'

print(f"Loading datasets from: {data_dir}")

train_dataset, val_dataset, test_dataset, metadata = load_preprocessed_datasets(data_dir)

print(f"\n✓ Datasets loaded successfully")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Validation samples: {len(val_dataset)}")
print(f"  Test samples: {len(test_dataset)}")
print(f"\n  Metadata:")
print(f"    Random seed: {metadata.get('random_seed', 'N/A')}")
print(f"    Train/Val/Test split: {metadata.get('train_split', 'N/A')}/{metadata.get('val_split', 'N/A')}/{metadata.get('test_split', 'N/A')}")

In [ ]:
# Create data loaders with configuration-based settings
batch_size = config['training']['batch_size']
num_workers = config['dataloader']['num_workers']
pin_memory = config['dataloader']['pin_memory']
persistent_workers = config['dataloader'].get('persistent_workers', False)

print(f"DataLoader settings:")
print(f"  Batch size: {batch_size}")
print(f"  Num workers: {num_workers}")
print(f"  Pin memory: {pin_memory}")
print(f"  Persistent workers: {persistent_workers}")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_variable_length,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers if num_workers > 0 else False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_variable_length,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers if num_workers > 0 else False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_variable_length,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

print(f"\n DataLoaders created successfully")
print(f"  Training batches: {len(train_loader)}")
print(f"  Validation batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

## 2. Initialize Model

The model is automatically created based on the configuration selected above.

In [ ]:
# Initialize model using factory function
# This automatically creates the correct model based on config['model']['type']
print(f"Creating model: {config['model']['type']}")
print(f"Hidden size: {config['model']['hidden_size']}")

model = create_model(config, device)

# Print model info
num_params = count_parameters(model)
print(f"\nTotal model parameters: {num_params:,}")

# Estimate and display GPU memory usage
if device == 'cuda':
    # Calculate rough memory estimate
    param_memory = num_params * 4 / 1024**3  # FP32 weights
    optimizer_memory = num_params * 8 / 1024**3  # AdamW states
    gradient_memory = num_params * 4 / 1024**3  # Gradients
    
    estimated_total = param_memory + optimizer_memory + gradient_memory
    
    print(f"\nEstimated GPU memory usage:")
    print(f"  Parameters: {param_memory:.2f} GB")
    print(f"  Optimizer states: {optimizer_memory:.2f} GB")
    print(f"  Gradients: {gradient_memory:.2f} GB")
    print(f"  Total (excluding activations): {estimated_total:.2f} GB")
    
    # Current memory usage
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    print(f"\nCurrent GPU memory:")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved: {reserved:.2f} GB")

## 3. Train Model

In [ ]:
# Set up checkpoint directory
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
checkpoint_dir = f'/content/{repo_name}/ai_model/models/checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

print(f"Checkpoint directory: {checkpoint_dir}")

# Clear GPU cache before training
if device == 'cuda':
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("Cleared GPU cache")

# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    device=device,
    checkpoint_dir=checkpoint_dir
)

# Display training summary
print("\n" + "=" * 60)
print("TRAINING SUMMARY")
print("=" * 60)
print(f"Model type: {config['model']['type']}")
print(f"Best validation loss: {history['best_val_loss']:.4f}")
print(f"Best validation accuracy: {history['best_val_accuracy']:.4f}")
print(f"Best epoch: {history['best_epoch'] + 1}")
print(f"Total epochs trained: {len(history['train_losses'])}")

# GPU memory usage
if device == 'cuda':
    peak_memory = torch.cuda.max_memory_allocated(0) / 1024**3
    print(f"\nPeak GPU memory used: {peak_memory:.2f} GB")

In [ ]:
# Visualize training progress
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss curves
axes[0].plot(history['train_losses'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_losses'], label='Val Loss', linewidth=2)
axes[0].axvline(x=history['best_epoch'], color='r', linestyle='--', alpha=0.5, label='Best Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curve
axes[1].plot(history['val_accuracies'], label='Val Accuracy', color='green', linewidth=2)
axes[1].axvline(x=history['best_epoch'], color='r', linestyle='--', alpha=0.5, label='Best Epoch')
axes[1].axhline(y=history['best_val_accuracy'], color='g', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'Validation Accuracy (Best: {history["best_val_accuracy"]:.4f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate curve
axes[2].plot(history['learning_rates'], label='Learning Rate', color='purple', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'/content/{repo_name}/ai_model/models/training_curves.png', dpi=150)
plt.show()

print(f"\nTraining curves saved to /content/{repo_name}/ai_model/models/training_curves.png")

## 4. Evaluate Model

In [ ]:
# Load best model checkpoint
from utils import load_checkpoint

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
best_model_path = f'/content/{repo_name}/ai_model/models/best_model.pth'

print(f"Loading best model from: {best_model_path}")

# Create a fresh model instance for loading
model = create_model(config, device)

checkpoint = load_checkpoint(
    best_model_path,
    model,
    device=device
)

print(f" Loaded best model")
print(f"  Model type: {config['model']['type']}")
print(f"  Epoch: {checkpoint['epoch'] + 1}")
print(f"  Validation loss: {checkpoint.get('val_loss', 'N/A')}")
print(f"  Validation accuracy: {checkpoint.get('val_accuracy', 'N/A')}")

In [ ]:
# Evaluate on test set
print("Evaluating model on test set...")

metrics = evaluate_model(
    model=model,
    test_loader=test_loader,
    device=device,
    confidence_threshold=config['trading']['confidence_threshold']
)

print("\n" + "=" * 60)
print("TEST SET METRICS")
print("=" * 60)
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Average Confidence: {metrics['avg_confidence']:.4f}")
print(f"\nPer-Class Metrics:")
for i, (prec, rec, f1) in enumerate(zip(metrics['precision'], metrics['recall'], metrics['f1_score'])):
    print(f"  Class {i}: Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
print(f"\nConfusion Matrix:")
print(metrics['confusion_matrix'])
print("=" * 60)

## 5. Comprehensive Backtest

In [ ]:
# Run comprehensive backtest
print("Running comprehensive backtest...")

backtest_results = comprehensive_backtest(
    model=model,
    test_dataset=test_dataset,
    device=device,
    confidence_threshold=config['trading']['confidence_threshold']
)

print("\n" + "=" * 60)
print("BACKTEST RESULTS")
print("=" * 60)
print(f"Total PnL: {backtest_results['total_pnl']:.4f} SOL")
print(f"Number of Tokens: {backtest_results['num_tokens']}")
print(f"Total Trades: {backtest_results['total_trades']}")
print(f"Overall Win Rate: {backtest_results['overall_win_rate']:.2%}")
print(f"Sharpe Ratio: {backtest_results['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {backtest_results['max_drawdown']:.2%}")
print(f"Avg Trades per Token: {backtest_results['avg_trades_per_token']:.2f}")
print(f"Avg PnL per Token: {backtest_results['avg_pnl_per_token']:.4f} SOL")
print("=" * 60)

## 6. Save Results

In [ ]:
import json
from datetime import datetime

# Save results summary
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
results_path = f'/content/{repo_name}/ai_model/models/results_summary.json'

results_summary = {
    'timestamp': datetime.now().isoformat(),
    'config': config,
    'training_history': {
        'best_epoch': int(history['best_epoch']),
        'best_val_loss': float(history['best_val_loss']),
    },
    'test_metrics': {
        'accuracy': float(metrics['accuracy']),
        'avg_confidence': float(metrics['avg_confidence']),
    },
    'backtest_results': {
        'total_pnl': float(backtest_results['total_pnl']),
        'num_tokens': int(backtest_results['num_tokens']),
        'total_trades': int(backtest_results['total_trades']),
        'win_rate': float(backtest_results['overall_win_rate']),
        'sharpe_ratio': float(backtest_results['sharpe_ratio']),
        'max_drawdown': float(backtest_results['max_drawdown']),
    }
}

os.makedirs(os.path.dirname(results_path), exist_ok=True)

with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"✓ Results saved to: {results_path}")

# Download results file
from google.colab import files
files.download(results_path)
print("✓ Results file downloaded to your computer")